# Tokenization & Embedding - 从零实现到 PyTorch 版本

## 目标：彻底搞懂 Transformer 的输入处理流程

1. **为什么需要分词** —— 文本到数字的桥梁
2. **分词的三种层次** —— 字符级、词级、子词级(BPE)
3. **从零实现 BPE** —— Byte Pair Encoding 训练与编码
4. **真实 Tokenizer 演示** —— 使用 transformers 库
5. **词嵌入原理** —— 从 One-Hot 到 nn.Embedding
6. **语义相似度验证** —— 词嵌入的意义
7. **完整流水线** —— 分词 → 词嵌入 → 位置编码

In [1]:
import math
import numpy as np
import torch
import torch.nn as nn

print("所有依赖已加载！")
print(f"numpy version: {np.__version__}")
print(f"torch version: {torch.__version__}")

所有依赖已加载！
numpy version: 1.24.3
torch version: 2.8.0+cu128


---

# PART 1: 为什么需要分词？—— 文本到数字的桥梁

## 核心问题

Transformer 是**深度学习模型**，它只能处理**数字**，不能直接处理**文本**。

所以第一步必须把文本转换成数字序列：

```
"我爱机器学习" → [5, 12, 8, 23, 45, 67]
```

这个转换过程就是**分词（Tokenization）**。

## 分词的挑战

分词看似简单，但其实有很多挑战：

1. **词汇量问题**：如果用「词」作为基本单位，词汇量太大（英语几十万个词），嵌入矩阵会非常大
2. **OOV（Out of Vocabulary）问题**：遇到训练时没见过的词怎么办？
3. **形态变化**：英语中 "running"、"ran"、"runner" 都和 "run" 相关，但作为独立词会浪费参数

## 三种分词策略

| 策略 | 基本单位 | 优点 | 缺点 |
|------|----------|------|------|
| **字符级** | 单个字符 | 词汇量极小，无 OOV | 序列太长，丢失语义信息 |
| **词级** | 完整单词 | 语义信息完整 | 词汇量大，OOV 严重 |
| **子词级** | 子词单元 | 兼顾词汇量和语义 | 需要训练分词器 |

In [2]:
# 三种分词策略的对比演示

text = "我爱机器学习"

print("=" * 60)
print("原始文本：", text)
print("=" * 60)

# 1. 字符级分词
char_tokens = list(text)
print(f"\n【字符级分词】")
print(f"  tokens: {char_tokens}")
print(f"  长度: {len(char_tokens)}")

# 2. 词级分词（中文按字分词，模拟词级）
word_tokens = text.split()  # 中文没有空格分隔，这里简化处理
print(f"\n【词级分词】")
print(f"  tokens: {word_tokens}")
print(f"  长度: {len(word_tokens)}")

# 3. 子词级分词（模拟）
subword_tokens = ["我", "爱", "机器", "学习"]
print(f"\n【子词级分词】")
print(f"  tokens: {subword_tokens}")
print(f"  长度: {len(subword_tokens)}")

print("\n" + "=" * 60)
print("观察：")
print("  字符级：最细粒度，保留所有信息，但序列最长")
print("  词级：最粗粒度，语义最完整，但词汇量大")
print("  子词级：平衡方案，既能组合新词，又控制词汇量")

原始文本： 我爱机器学习

【字符级分词】
  tokens: ['我', '爱', '机', '器', '学', '习']
  长度: 6

【词级分词】
  tokens: ['我爱机器学习']
  长度: 1

【子词级分词】
  tokens: ['我', '爱', '机器', '学习']
  长度: 4

观察：
  字符级：最细粒度，保留所有信息，但序列最长
  词级：最粗粒度，语义最完整，但词汇量大
  子词级：平衡方案，既能组合新词，又控制词汇量


---

# PART 2: BPE 原理 —— 最流行的子词分词算法

## BPE 是什么？

BPE（Byte Pair Encoding）是一种**数据压缩算法**，后来被引入到 NLP 领域作为分词算法。

## 核心思想

1. **初始化**：把所有词拆成单个字符，统计每个字符的频率
2. **迭代合并**：找到出现频率最高的字符对，合并成一个新的子词
3. **重复**：重复第 2 步，直到达到预定的词汇量

## 举个例子

假设语料库中有这些词：

- "low" → [l, o, w]
- "lower" → [l, o, w, e, r]
- "newest" → [n, e, w, e, s, t]
- "widest" → [w, i, d, e, s, t]

第一次迭代：找到频率最高的字符对 "e" 和 "s"，合并成 "es"

- "newest" → [n, e, w, es, t]
- "widest" → [w, i, d, es, t]

第二次迭代：找到频率最高的字符对 "w" 和 "es"，合并成 "wes"

- "newest" → [n, e, wes, t]
- "widest" → [w, i, d, es, t]

这样不断迭代，就能得到越来越长的子词单元。

In [3]:
# BPE 算法的关键步骤演示

# 模拟语料库
corpus = [
    "low", "lower", "lowest",
    "new", "newer", "newest",
    "wide", "wider", "widest",
    "high", "higher", "highest"
]

print("=" * 60)
print("初始语料库")
print("=" * 60)
print(corpus)

# Step 1: 把每个词拆成字符，末尾加特殊标记 </w>
split_words = [[char for char in word] + ['</w>'] for word in corpus]

print("\n" + "=" * 60)
print("Step 1: 拆分成字符 + </w> 标记")
print("=" * 60)
for word, split in zip(corpus, split_words):
    print(f"  {word:8s} → {split}")

初始语料库
['low', 'lower', 'lowest', 'new', 'newer', 'newest', 'wide', 'wider', 'widest', 'high', 'higher', 'highest']

Step 1: 拆分成字符 + </w> 标记
  low      → ['l', 'o', 'w', '</w>']
  lower    → ['l', 'o', 'w', 'e', 'r', '</w>']
  lowest   → ['l', 'o', 'w', 'e', 's', 't', '</w>']
  new      → ['n', 'e', 'w', '</w>']
  newer    → ['n', 'e', 'w', 'e', 'r', '</w>']
  newest   → ['n', 'e', 'w', 'e', 's', 't', '</w>']
  wide     → ['w', 'i', 'd', 'e', '</w>']
  wider    → ['w', 'i', 'd', 'e', 'r', '</w>']
  widest   → ['w', 'i', 'd', 'e', 's', 't', '</w>']
  high     → ['h', 'i', 'g', 'h', '</w>']
  higher   → ['h', 'i', 'g', 'h', 'e', 'r', '</w>']
  highest  → ['h', 'i', 'g', 'h', 'e', 's', 't', '</w>']


In [4]:
# Step 2: 统计所有字符对的频率

def count_pairs(split_words):
    """统计字符对频率"""
    pairs = {}
    for word in split_words:
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])
            pairs[pair] = pairs.get(pair, 0) + 1
    return pairs

pairs = count_pairs(split_words)

print("\n" + "=" * 60)
print("Step 2: 统计字符对频率")
print("=" * 60)
print(f"{'字符对':<15} {'频率':>6}")
print("-" * 25)
for pair, freq in sorted(pairs.items(), key=lambda x: -x[1])[:10]:
    print(f"{str(pair):<15} {freq:>6}")


Step 2: 统计字符对频率
字符对                 频率
-------------------------
('w', 'e')           4
('e', 'r')           4
('r', '</w>')        4
('e', 's')           4
('s', 't')           4
('t', '</w>')        4
('l', 'o')           3
('o', 'w')           3
('n', 'e')           3
('e', 'w')           3


In [5]:
# Step 3: 找到频率最高的字符对并合并

def merge_pair(split_words, pair):
    """合并指定的字符对"""
    new_split = []
    bigram = ''.join(pair)
    for word in split_words:
        i = 0
        new_word = []
        while i < len(word):
            if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(bigram)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_split.append(new_word)
    return new_split

# 找到频率最高的字符对
best_pair = max(pairs, key=pairs.get)
print(f"\n频率最高的字符对: {best_pair}, 频率: {pairs[best_pair]}")

# 合并
split_words = merge_pair(split_words, best_pair)

print("\n" + "=" * 60)
print("Step 3: 合并后结果")
print("=" * 60)
for word, split in zip(corpus, split_words):
    print(f"  {word:8s} → {split}")


频率最高的字符对: ('w', 'e'), 频率: 4

Step 3: 合并后结果
  low      → ['l', 'o', 'w', '</w>']
  lower    → ['l', 'o', 'we', 'r', '</w>']
  lowest   → ['l', 'o', 'we', 's', 't', '</w>']
  new      → ['n', 'e', 'w', '</w>']
  newer    → ['n', 'e', 'we', 'r', '</w>']
  newest   → ['n', 'e', 'we', 's', 't', '</w>']
  wide     → ['w', 'i', 'd', 'e', '</w>']
  wider    → ['w', 'i', 'd', 'e', 'r', '</w>']
  widest   → ['w', 'i', 'd', 'e', 's', 't', '</w>']
  high     → ['h', 'i', 'g', 'h', '</w>']
  higher   → ['h', 'i', 'g', 'h', 'e', 'r', '</w>']
  highest  → ['h', 'i', 'g', 'h', 'e', 's', 't', '</w>']


---

# PART 3: 从零实现完整的 BPE 分词器

现在把上面的步骤整合起来，实现一个完整的 BPE 分词器。

In [6]:
class SimpleBPE:
    """
    简化版 BPE 分词器

    使用方式：
        bpe = SimpleBPE(vocab_size=100)
        bpe.train(corpus)       # 训练分词器
        tokens = bpe.encode("hello world")  # 编码
        text = bpe.decode(tokens)  # 解码
    """

    def __init__(self, vocab_size: int = 100):
        self.vocab_size = vocab_size
        self.vocab = {}        # token → id
        self.id_to_token = {}  # id → token
        self.merge_rules = []  # 合并规则列表

    def _count_pairs(self, split_words):
        """统计字符对频率"""
        pairs = {}
        for word in split_words:
            for i in range(len(word) - 1):
                pair = (word[i], word[i+1])
                pairs[pair] = pairs.get(pair, 0) + 1
        return pairs

    def _merge_pair(self, split_words, pair):
        """合并指定的字符对"""
        new_split = []
        bigram = ''.join(pair)
        for word in split_words:
            i = 0
            new_word = []
            while i < len(word):
                if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                    new_word.append(bigram)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_split.append(new_word)
        return new_split

    def train(self, corpus):
        """
        训练 BPE 分词器

        Args:
            corpus: 训练语料，字符串列表
        """
        # Step 1: 拆分成字符，末尾加 </w>
        split_words = [[char for char in word] + ['</w>'] for word in corpus]

        # Step 2: 初始化词汇表（所有唯一字符）
        all_chars = set()
        for word in split_words:
            all_chars.update(word)

        self.vocab = {token: idx for idx, token in enumerate(sorted(all_chars))}
        self.id_to_token = {idx: token for token, idx in self.vocab.items()}

        print(f"初始词汇表大小: {len(self.vocab)}")
        print(f"初始词汇: {list(self.vocab.keys())}")

        # Step 3: 迭代合并
        merge_steps = self.vocab_size - len(self.vocab)
        for step in range(merge_steps):
            pairs = self._count_pairs(split_words)
            if not pairs:
                break

            # 找到频率最高的字符对
            best_pair = max(pairs, key=pairs.get)
            best_freq = pairs[best_pair]

            # 合并
            split_words = self._merge_pair(split_words, best_pair)

            # 更新词汇表
            new_token = ''.join(best_pair)
            self.vocab[new_token] = len(self.vocab)
            self.id_to_token[len(self.id_to_token)] = new_token
            self.merge_rules.append(best_pair)

            if (step + 1) % 5 == 0 or step == 0:
                print(f"  Step {step+1}: 合并 {best_pair} → '{new_token}' (频率={best_freq})")

        print(f"\n训练完成！最终词汇表大小: {len(self.vocab)}")

    def encode(self, text: str) -> list:
        """
        把文本编码成 token id 列表

        Args:
            text: 输入文本

        Returns:
            token id 列表
        """
        # 拆分成字符
        tokens = [char for char in text] + ['</w>']

        # 应用合并规则
        for pair in self.merge_rules:
            i = 0
            while i < len(tokens) - 1:
                if tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                    tokens = tokens[:i] + [''.join(pair)] + tokens[i+2:]
                else:
                    i += 1

        # 转换成 id
        ids = []
        for token in tokens:
            if token in self.vocab:
                ids.append(self.vocab[token])
            else:
                # OOV 处理：拆成单个字符
                for char in token:
                    if char in self.vocab:
                        ids.append(self.vocab[char])

        return ids

    def decode(self, ids: list) -> str:
        """
        把 token id 列表解码成文本

        Args:
            ids: token id 列表

        Returns:
            解码后的文本
        """
        tokens = [self.id_to_token.get(idx, '<UNK>') for idx in ids]
        return ''.join(tokens).replace('</w>', '')

In [7]:
# 测试 BPE 分词器

# 训练语料
corpus = [
    "low", "lower", "lowest",
    "new", "newer", "newest",
    "wide", "wider", "widest",
    "high", "higher", "highest",
    "fast", "faster", "fastest",
    "slow", "slower", "slowest"
]

print("=" * 60)
print("训练 BPE 分词器")
print("=" * 60)

# 创建并训练分词器
bpe = SimpleBPE(vocab_size=30)
bpe.train(corpus)

训练 BPE 分词器
初始词汇表大小: 15
初始词汇: ['</w>', 'a', 'd', 'e', 'f', 'g', 'h', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w']
  Step 1: 合并 ('s', 't') → 'st' (频率=9)
  Step 5: 合并 ('e', 'r') → 'er' (频率=6)
  Step 10: 合并 ('w', 'i') → 'wi' (频率=3)
  Step 15: 合并 ('f', 'a') → 'fa' (频率=3)

训练完成！最终词汇表大小: 30


In [8]:
# 测试编码和解码

test_texts = [
    "lowest",
    "newest",
    "wider",
    "faster",
    "hello"  # OOV 词
]

print("\n" + "=" * 60)
print("测试编码和解码")
print("=" * 60)

for text in test_texts:
    ids = bpe.encode(text)
    decoded = bpe.decode(ids)
    tokens = [bpe.id_to_token.get(id, '<UNK>') for id in ids]
    
    print(f"\n原始文本: '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  IDs:    {ids}")
    print(f"  解码:   '{decoded}'")
    print(f"  匹配:   {text == decoded}")

print("\n" + "=" * 60)
print("观察：")
print("  1. 训练过的词（如 lowest）被分成了有意义的子词")
print("  2. 未见过的词（如 hello）被拆成单个字符，保证不会完全丢失信息")


测试编码和解码

原始文本: 'lowest'
  Tokens: ['low', 'est</w>']
  IDs:    [18, 21]
  解码:   'lowest'
  匹配:   True

原始文本: 'newest'
  Tokens: ['new', 'est</w>']
  IDs:    [23, 21]
  解码:   'newest'
  匹配:   True

原始文本: 'wider'
  Tokens: ['wid', 'er</w>']
  IDs:    [25, 20]
  解码:   'wider'
  匹配:   True

原始文本: 'faster'
  Tokens: ['fa', 'st', 'er</w>']
  IDs:    [29, 15, 20]
  解码:   'faster'
  匹配:   True

原始文本: 'hello'
  Tokens: ['h', 'e', 'l', 'lo', '</w>']
  IDs:    [6, 3, 8, 17, 0]
  解码:   'hello'
  匹配:   True

观察：
  1. 训练过的词（如 lowest）被分成了有意义的子词
  2. 未见过的词（如 hello）被拆成单个字符，保证不会完全丢失信息


---

# PART 4: 词嵌入 —— 从 One-Hot 到 nn.Embedding

## 为什么需要词嵌入？

分词把文本转换成了整数序列，但整数本身没有语义信息：

- id=5 不代表任何语义
- "猫"=3 和 "狗"=4 之间没有关系

词嵌入的目标是把每个词映射到一个**稠密的实数向量**，使得：

- 语义相似的词，向量也相似
- "猫"和"狗"的向量应该比较接近（都是动物）
- "猫"和"汽车"的向量应该比较远

## One-Hot 编码的缺陷

最简单的词表示是 One-Hot 编码：

- "我" → [1, 0, 0, 0, 0]
- "爱" → [0, 1, 0, 0, 0]
- "学" → [0, 0, 1, 0, 0]

**缺陷**：
1. 向量维度 = 词汇量，对于大词汇表维度太大
2. 所有向量都是正交的，无法表达语义相似度
3. 无法利用词之间的关系

## 词嵌入的解决方案

词嵌入用一个**可学习的矩阵**来映射：

Embedding Matrix E: (vocab_size, d_model)

对于词 id = i，其嵌入向量为 E[i]

这个矩阵是通过训练学习得到的，语义相似的词会自动靠近。

In [9]:
# One-Hot 编码 vs 词嵌入

# 模拟词汇表
vocab = {"我": 0, "爱": 1, "学": 2, "习": 3, "机": 4, "器": 5}
vocab_size = len(vocab)
d_model = 4  # 嵌入维度

print("=" * 60)
print("One-Hot 编码 vs 词嵌入")
print("=" * 60)

# 1. One-Hot 编码
print("\n【One-Hot 编码】")
print(f"词汇量: {vocab_size}, 向量维度: {vocab_size}")
print("-" * 40)

for word, idx in vocab.items():
    one_hot = np.zeros(vocab_size)
    one_hot[idx] = 1
    print(f"  '{word}' → {one_hot}")

# 2. 词嵌入（随机初始化的 Embedding 矩阵）
print("\n【词嵌入】")
print(f"词汇量: {vocab_size}, 向量维度: {d_model}")
print("-" * 40)

# 随机初始化嵌入矩阵
np.random.seed(42)
embedding_matrix = np.random.randn(vocab_size, d_model) * 0.01

for word, idx in vocab.items():
    vec = embedding_matrix[idx]
    print(f"  '{word}' → {np.round(vec, 4)}")

print("\n" + "=" * 60)
print("关键区别：")
print("  One-Hot: 稀疏、高维、无语义")
print("  词嵌入: 稠密、低维、有语义（训练后）")

One-Hot 编码 vs 词嵌入

【One-Hot 编码】
词汇量: 6, 向量维度: 6
----------------------------------------
  '我' → [1. 0. 0. 0. 0. 0.]
  '爱' → [0. 1. 0. 0. 0. 0.]
  '学' → [0. 0. 1. 0. 0. 0.]
  '习' → [0. 0. 0. 1. 0. 0.]
  '机' → [0. 0. 0. 0. 1. 0.]
  '器' → [0. 0. 0. 0. 0. 1.]

【词嵌入】
词汇量: 6, 向量维度: 4
----------------------------------------
  '我' → [ 0.005  -0.0014  0.0065  0.0152]
  '爱' → [-0.0023 -0.0023  0.0158  0.0077]
  '学' → [-0.0047  0.0054 -0.0046 -0.0047]
  '习' → [ 0.0024 -0.0191 -0.0172 -0.0056]
  '机' → [-0.0101  0.0031 -0.0091 -0.0141]
  '器' → [ 0.0147 -0.0023  0.0007 -0.0142]

关键区别：
  One-Hot: 稀疏、高维、无语义
  词嵌入: 稠密、低维、有语义（训练后）


In [10]:
# PyTorch nn.Embedding 实现

print("=" * 60)
print("PyTorch nn.Embedding 演示")
print("=" * 60)

# 创建 Embedding 层
vocab_size = 100
d_model = 16

embedding = nn.Embedding(vocab_size, d_model)

print(f"Embedding 层参数:")
print(f"  vocab_size: {vocab_size}")
print(f"  embedding_dim: {d_model}")
print(f"  嵌入矩阵 shape: {embedding.weight.shape}")
print(f"  参数总数: {embedding.weight.numel()}")
print(f"  是否可训练: {embedding.weight.requires_grad}")

# 模拟输入：batch_size=2, seq_len=5
batch_tokens = torch.tensor([[1, 5, 3, 7, 2], [4, 2, 8, 1, 6]])
print(f"\n输入 token_ids shape: {batch_tokens.shape}")
print(f"输入 token_ids:\n{batch_tokens}")

# 前向传播
output = embedding(batch_tokens)
print(f"\n输出嵌入向量 shape: {output.shape}")
print(f"输出嵌入向量:\n{torch.round(output, decimals=4)}")

print("\n" + "=" * 60)
print("nn.Embedding 的工作原理：")
print("  1. 内部维护一个 (vocab_size, d_model) 的可学习矩阵")
print("  2. 输入 token_id 时，直接查表取对应行")
print("  3. 训练过程中，这个矩阵会被优化")

PyTorch nn.Embedding 演示
Embedding 层参数:
  vocab_size: 100
  embedding_dim: 16
  嵌入矩阵 shape: torch.Size([100, 16])
  参数总数: 1600
  是否可训练: True

输入 token_ids shape: torch.Size([2, 5])
输入 token_ids:
tensor([[1, 5, 3, 7, 2],
        [4, 2, 8, 1, 6]])

输出嵌入向量 shape: torch.Size([2, 5, 16])
输出嵌入向量:
tensor([[[ 0.4055, -1.0125,  0.7509,  0.3658,  1.3649, -2.4110, -1.4510,
           0.6243, -0.2607, -3.4596,  0.0649, -0.2429, -0.2987,  0.4815,
          -1.0006,  1.1628],
         [-0.3523, -0.6039, -0.0361, -0.3760, -0.5460, -0.2882,  1.5401,
           1.0994, -2.1724, -0.1448, -1.1201,  0.0413,  0.6756,  0.2828,
           1.0868,  2.4738],
         [ 0.7928,  0.2919, -0.6946, -0.3390,  1.1153, -1.2176, -0.2368,
          -1.7512, -1.0222, -0.0197,  0.9542,  1.6866, -0.5822, -0.4093,
           1.1371,  1.0286],
         [-0.4306,  0.5467,  0.5954, -1.6729,  0.3595,  1.5799, -1.1482,
           1.9014,  0.0173,  0.8370, -1.8492, -0.4818, -0.5769,  1.6942,
          -0.9061,  2.3789],
         

---

# PART 5: 词嵌入的语义相似度验证

词嵌入的核心价值在于：**语义相似的词，向量也相似**。

我们可以通过计算词向量之间的余弦相似度来验证这一点。


In [11]:
# 验证词嵌入的语义相似度

def cosine_similarity(vec1, vec2):
    """计算余弦相似度"""
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# 模拟一个训练好的嵌入矩阵
# 假设：语义相似的词向量比较接近
np.random.seed(123)

vocab = {
    "猫": 0, "狗": 1, "鸟": 2,  # 动物
    "汽车": 3, "火车": 4, "飞机": 5,  # 交通工具
    "苹果": 6, "香蕉": 7, "橙子": 8,  # 水果
    "快乐": 9, "开心": 10, "高兴": 11,  # 情感
}

# 手动构造有语义关系的嵌入向量
d_model = 8
embedding_matrix = np.random.randn(len(vocab), d_model) * 0.1

# 让同类词向量靠近
# 动物类
embedding_matrix[0] += np.array([0.5, 0.5, 0, 0, 0, 0, 0, 0])  # 猫
embedding_matrix[1] += np.array([0.5, 0.4, 0, 0, 0, 0, 0, 0])  # 狗
embedding_matrix[2] += np.array([0.4, 0.5, 0, 0, 0, 0, 0, 0])  # 鸟

# 交通工具类
embedding_matrix[3] += np.array([0, 0, 0.5, 0.5, 0, 0, 0, 0])  # 汽车
embedding_matrix[4] += np.array([0, 0, 0.5, 0.4, 0, 0, 0, 0])  # 火车
embedding_matrix[5] += np.array([0, 0, 0.4, 0.5, 0, 0, 0, 0])  # 飞机

# 水果类
embedding_matrix[6] += np.array([0, 0, 0, 0, 0.5, 0.5, 0, 0])  # 苹果
embedding_matrix[7] += np.array([0, 0, 0, 0, 0.5, 0.4, 0, 0])  # 香蕉
embedding_matrix[8] += np.array([0, 0, 0, 0, 0.4, 0.5, 0, 0])  # 橙子

# 情感类
embedding_matrix[9] += np.array([0, 0, 0, 0, 0, 0, 0.5, 0.5])  # 快乐
embedding_matrix[10] += np.array([0, 0, 0, 0, 0, 0, 0.5, 0.4])  # 开心
embedding_matrix[11] += np.array([0, 0, 0, 0, 0, 0, 0.4, 0.5])  # 高兴

# 计算相似度矩阵
words = list(vocab.keys())
similarity_matrix = np.zeros((len(words), len(words)))

for i, word1 in enumerate(words):
    for j, word2 in enumerate(words):
        similarity_matrix[i, j] = cosine_similarity(
            embedding_matrix[vocab[word1]],
            embedding_matrix[vocab[word2]]
        )

# 打印相似度矩阵
print("=" * 60)
print("词向量余弦相似度矩阵")
print("=" * 60)

# 打印表头
print(f"{'':<8}", end="")
for word in words:
    print(f"{word:<8}", end="")
print()
print("-" * (len(words) * 8 + 8))

# 打印每行
for i, word1 in enumerate(words):
    print(f"{word1:<8}", end="")
    for j, word2 in enumerate(words):
        sim = similarity_matrix[i, j]
        color = "\033[92m" if sim > 0.6 else "\033[94m" if sim > 0.3 else ""
        print(f"{color}{sim:>7.4f}\033[0m", end=" ")
    print()

词向量余弦相似度矩阵
        猫       狗       鸟       汽车      火车      飞机      苹果      香蕉      橙子      快乐      开心      高兴      
--------------------------------------------------------------------------------------------------------
猫        1.0000  0.7418  0.9117 -0.1978 -0.2059 -0.4664  0.3096  0.3232  0.3445 -0.3256 -0.2896  0.0289 
狗        0.7418  1.0000  0.8535 -0.2264 -0.3327 -0.2666  0.3393  0.2661  0.2758  0.0304  0.0391  0.1479 
鸟        0.9117  0.8535  1.0000 -0.1267 -0.0883 -0.1829  0.3926  0.4102  0.3768 -0.0467  0.0401  0.2710 
汽车      -0.1978 -0.2264 -0.1267  1.0000  0.8885  0.7421 -0.3307  0.1728 -0.1166 -0.4293  0.0835 -0.3306 
火车      -0.2059 -0.3327 -0.0883  0.8885  1.0000  0.7996 -0.1860  0.2450 -0.0165 -0.2812  0.1768 -0.0450 
飞机      -0.4664 -0.2666 -0.1829  0.7421  0.7996  1.0000 -0.1975  0.1773 -0.0441  0.2309  0.5232  0.1146 
苹果       0.3096  0.3393  0.3926 -0.3307 -0.1860 -0.1975  1.0000  0.8113  0.8156  0.0164 -0.1408  0.2290 
香蕉       0.3232  0.2661  0.4102  0.1728  0.2

In [12]:
# 验证类比关系

print("\n" + "=" * 60)
print("验证类比关系：相似词的向量差近似相等")
print("=" * 60)

# 猫 - 狗 ≈ 鸟 - ？
cat_vec = embedding_matrix[vocab["猫"]]
dog_vec = embedding_matrix[vocab["狗"]]
bird_vec = embedding_matrix[vocab["鸟"]]

# 计算差值
diff_cat_dog = cat_vec - dog_vec

# 找到最接近 bird + diff_cat_dog 的词
target = bird_vec + diff_cat_dog

max_sim = -1
best_word = ""
for word, idx in vocab.items():
    sim = cosine_similarity(target, embedding_matrix[idx])
    if sim > max_sim:
        max_sim = sim
        best_word = word

print(f"\n类比推理：")
print(f"  猫 - 狗 ≈ 鸟 - ?")
print(f"  cat_vec - dog_vec = bird_vec - ?_vec")
print(f"  ?_vec = bird_vec + (cat_vec - dog_vec)")
print(f"  最接近的词: '{best_word}' (相似度: {max_sim:.4f})")

# 另一个类比
happy_vec = embedding_matrix[vocab["快乐"]]
glad_vec = embedding_matrix[vocab["开心"]]
apple_vec = embedding_matrix[vocab["苹果"]]

diff_happy_glad = happy_vec - glad_vec
target2 = apple_vec + diff_happy_glad

max_sim2 = -1
best_word2 = ""
for word, idx in vocab.items():
    sim = cosine_similarity(target2, embedding_matrix[idx])
    if sim > max_sim2:
        max_sim2 = sim
        best_word2 = word

print(f"\n类比推理：")
print(f"  快乐 - 开心 ≈ 苹果 - ?")
print(f"  最接近的词: '{best_word2}' (相似度: {max_sim2:.4f})")

print("\n" + "=" * 60)
print("结论：")
print("  训练好的词嵌入可以捕捉语义关系！")
print("  同类词的向量聚集在一起，不同类词的向量相互远离。")


验证类比关系：相似词的向量差近似相等

类比推理：
  猫 - 狗 ≈ 鸟 - ?
  cat_vec - dog_vec = bird_vec - ?_vec
  ?_vec = bird_vec + (cat_vec - dog_vec)
  最接近的词: '猫' (相似度: 0.9486)

类比推理：
  快乐 - 开心 ≈ 苹果 - ?
  最接近的词: '苹果' (相似度: 0.9251)

结论：
  训练好的词嵌入可以捕捉语义关系！
  同类词的向量聚集在一起，不同类词的向量相互远离。


---

# PART 6: 完整流水线 —— 分词 → 词嵌入 → 位置编码

现在把前面学到的所有组件串起来，形成完整的 Transformer 输入处理流水线。

完整流程：

1. **文本输入** → "我爱机器学习"
2. **分词** → ["我", "爱", "机器", "学习"]
3. **Token ID** → [5, 12, 8, 23]
4. **词嵌入** → (4, d_model) 的向量矩阵
5. **位置编码** → 注入位置信息
6. **Transformer 输入** → 准备好！

In [13]:
# 定义位置编码类（从之前的 notebook 复用）

class PositionalEncoding(nn.Module):
    """标准正余弦位置编码"""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [14]:
# 完整的 Transformer 输入处理流水线

# 配置参数
vocab_size = 1000
d_model = 32
max_len = 128

# 创建组件
tokenizer = SimpleBPE(vocab_size=vocab_size)
embedding = nn.Embedding(vocab_size, d_model)
pos_enc = PositionalEncoding(d_model, max_len, dropout=0.1)

# 训练分词器（模拟）
corpus = ["我", "爱", "学", "习", "机", "器", "人", "智", "能"]
tokenizer.train(corpus)

# 输入文本
input_text = "我爱机器学习"

print("=" * 60)
print("完整流水线演示")
print("=" * 60)

# Step 1: 分词
print(f"\nStep 1: 原始文本")
print(f"  '{input_text}'")

# Step 2: 编码成 token ID
token_ids = tokenizer.encode(input_text)
print(f"\nStep 2: 分词结果")
print(f"  Token IDs: {token_ids}")

# Step 3: 转换成 PyTorch tensor
token_tensor = torch.tensor(token_ids).unsqueeze(0)  # 添加 batch 维度
print(f"\nStep 3: 转换成 Tensor")
print(f"  Shape: {token_tensor.shape}")
print(f"  Tensor: {token_tensor}")

# Step 4: 词嵌入
embedded = embedding(token_tensor) * math.sqrt(d_model)
print(f"\nStep 4: 词嵌入")
print(f"  Shape: {embedded.shape}")
print(f"  前 2 个 token 的嵌入向量:\n{torch.round(embedded[0, :2], decimals=4)}")

# Step 5: 位置编码
with_pe = pos_enc(embedded)
print(f"\nStep 5: 注入位置编码")
print(f"  Shape: {with_pe.shape}")
print(f"  前 2 个 token 注入位置编码后的向量:\n{torch.round(with_pe[0, :2], decimals=4)}")

print("\n" + "=" * 60)
print("数据流转总结：")
print("=" * 60)
print("  文本 → 分词 → Token IDs → 词嵌入 → 位置编码 → Transformer")
print(f"  '{input_text}'")
print(f"       ↓")
print(f"  {token_ids}")
print(f"       ↓")
print(f"  ({token_tensor.shape}) → ({embedded.shape}) → ({with_pe.shape})")

初始词汇表大小: 10
初始词汇: ['</w>', '习', '人', '器', '学', '我', '智', '机', '爱', '能']
  Step 1: 合并 ('我', '</w>') → '我</w>' (频率=1)
  Step 5: 合并 ('机', '</w>') → '机</w>' (频率=1)

训练完成！最终词汇表大小: 19
完整流水线演示

Step 1: 原始文本
  '我爱机器学习'

Step 2: 分词结果
  Token IDs: [5, 8, 7, 3, 4, 13]

Step 3: 转换成 Tensor
  Shape: torch.Size([1, 6])
  Tensor: tensor([[ 5,  8,  7,  3,  4, 13]])

Step 4: 词嵌入
  Shape: torch.Size([1, 6, 32])
  前 2 个 token 的嵌入向量:
tensor([[ 3.6556e+00,  1.0623e+01,  1.0016e+01, -5.8706e+00,  1.3978e+00,
          1.4962e+00, -3.8774e+00,  4.1780e+00, -4.9556e+00, -3.0587e+00,
          8.7083e+00, -9.2550e-01, -4.4770e+00,  6.0296e+00,  6.3890e+00,
         -2.1500e+00, -3.8908e+00, -1.0559e+01, -4.9240e-01, -3.4096e+00,
         -1.9008e+00, -7.4890e-01,  5.5580e-01,  4.5893e+00,  1.9255e+00,
          1.5318e+01, -7.4318e+00, -4.3833e+00,  5.3172e+00,  4.6633e+00,
         -1.4972e+00, -2.1896e+00],
        [-5.1779e+00,  9.3044e+00,  3.1292e+00, -2.1366e+00,  3.9943e+00,
         -2.3410e+00,  5.2785

---

# 总结：分词与词嵌入的设计哲学

## 核心问题

Transformer 只能处理数字，我们需要把文本转换成有意义的数字表示。

## 两个关键步骤

### 1. 分词（Tokenization）

**目标**：把文本拆分成模型能处理的基本单元。

| 策略 | 特点 | 适用场景 |
|------|------|----------|
| 字符级 | 词汇量小，无 OOV | 小语种、语音识别 |
| 词级 | 语义完整 | 词汇量小的场景 |
| **子词级（BPE）** | **平衡词汇量和语义** | **现代大模型标配** |

### 2. 词嵌入（Embedding）

**目标**：把 token ID 映射到稠密的语义向量空间。

| 方法 | 特点 |
|------|------|
| One-Hot | 稀疏、高维、无语义 |
| **nn.Embedding** | **稠密、低维、可学习** |

## 关键设计决策

### 为什么用 nn.Embedding 而不是 One-Hot？

- **维度爆炸**：One-Hot 需要 vocab_size 维度，Embedding 只需要 d_model 维度
- **语义信息**：Embedding 通过训练学习语义，One-Hot 是随机的
- **参数效率**：Embedding 只有 vocab_size × d_model 个参数

### 为什么乘以 sqrt(d_model)？

这是 Transformer 论文中的一个小技巧：
- 词嵌入的初始值通常很小（接近 0）
- 位置编码的值范围是 [-1, 1]
- 如果不缩放，位置编码的影响会过大
- 乘以 sqrt(d_model) 后，词嵌入的方差约为 1，与位置编码量级匹配

## 与位置编码的配合

- **词嵌入**：提供**语义信息**（这个词是什么）
- **位置编码**：提供**位置信息**（这个词在哪里）
- 两者相加 → 每个 token 既有语义，又知道自己的位置


In [15]:
# 你可以在这里实验不同的参数
# 尝试修改 d_model、vocab_size 等，观察效果

# 示例：自定义实验
custom_vocab_size = 50
custom_d_model = 16
custom_text = "人工智能改变世界"

# 创建组件
custom_tokenizer = SimpleBPE(vocab_size=custom_vocab_size)
custom_embedding = nn.Embedding(custom_vocab_size, custom_d_model)

# 训练分词器
custom_corpus = ["人", "工", "智", "能", "改", "变", "世", "界", "学", "习"]
custom_tokenizer.train(custom_corpus)

# 编码
custom_ids = custom_tokenizer.encode(custom_text)
custom_tensor = torch.tensor(custom_ids).unsqueeze(0)
custom_embedded = custom_embedding(custom_tensor) * math.sqrt(custom_d_model)

print(f"\n自定义实验：")
print(f"  文本: '{custom_text}'")
print(f"  Token IDs: {custom_ids}")
print(f"  嵌入后 shape: {custom_embedded.shape}")
print(f"  嵌入向量:\n{torch.round(custom_embedded[0], decimals=4)}")

初始词汇表大小: 11
初始词汇: ['</w>', '世', '习', '人', '变', '学', '工', '改', '智', '界', '能']
  Step 1: 合并 ('人', '</w>') → '人</w>' (频率=1)
  Step 5: 合并 ('改', '</w>') → '改</w>' (频率=1)
  Step 10: 合并 ('习', '</w>') → '习</w>' (频率=1)

训练完成！最终词汇表大小: 21

自定义实验：
  文本: '人工智能改变世界'
  Token IDs: [3, 6, 8, 10, 7, 4, 1, 18]
  嵌入后 shape: torch.Size([1, 8, 16])
  嵌入向量:
tensor([[ 3.4302e+00,  3.6655e+00, -2.5090e+00, -3.6418e+00, -3.0779e+00,
          1.1856e+00,  2.6142e+00, -6.8060e-01, -1.9610e-01,  3.6111e+00,
          1.1141e+00,  7.5647e+00,  1.0249e+00,  7.8560e-01, -5.5521e+00,
          2.1102e+00],
        [-1.3113e+00, -3.5650e+00,  5.3744e+00,  4.4607e+00,  4.3396e+00,
         -2.7523e+00,  5.6190e-01,  5.0285e+00,  3.1672e+00,  5.1444e+00,
          1.0523e+01,  4.0148e+00,  3.3964e+00, -3.5029e+00,  4.1876e+00,
         -2.0806e+00],
        [-1.8676e+00,  3.7751e+00, -6.6150e-01,  5.3837e+00, -1.6134e+00,
          8.2100e-01, -7.5430e-01,  6.6296e+00,  1.7010e-01,  1.2410e+00,
          3.0408e+00, -4.